In [20]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import geopandas as gpd
from skimpy import clean_columns

## LA County 2020 Census Tracts Data

In [21]:
# read in LA county census tracts data
cts = gpd.read_file('../../../data/raw/LA_County_2020_Census_Tracts.geojson')

In [22]:
# preview
cts.head()

,OBJECTID,CT20,LABEL,ShapeSTArea,ShapeSTLength,geometry
0,4992,101110,1011.10,1.229562e+07,15083.854287,"POLYGON ((-118.29793 34.26323, -118.30082 34.2..."
1,4993,101122,1011.22,2.845774e+07,31671.455844,"POLYGON ((-118.27743 34.25991, -118.27743 34.2..."
2,4994,101220,1012.20,7.522093e+06,12698.783810,"POLYGON ((-118.27818 34.25577, -118.27887 34.2..."
3,4995,101221,1012.21,3.812000e+06,9161.710543,"POLYGON ((-118.28735 34.25591, -118.28863 34.2..."
4,4996,101222,1012.22,3.191371e+06,9980.600461,"POLYGON ((-118.28594 34.2559, -118.28697 34.25..."


In [23]:
# check crs
cts.crs

<Geographic 2D CRS: EPSG:4326>
Name: WGS 84
Axis Info [ellipsoidal]:
- Lat[north]: Geodetic latitude (degree)
- Lon[east]: Geodetic longitude (degree)
Area of Use:
- name: World.
- bounds: (-180.0, -90.0, 180.0, 90.0)
Datum: World Geodetic System 1984 ensemble
- Ellipsoid: WGS 84
- Prime Meridian: Greenwich

In [24]:
# drop unnecessary columns
cts = cts.drop(columns = [
    'OBJECTID',
    'CT20',
    'ShapeSTLength',
    'ShapeSTArea'
])

In [25]:
# rename some columns
cts = cts.rename(columns = {
    'LABEL': 'TRACT'
})

In [26]:
# set TRACT values as float
cts['TRACT'] = cts['TRACT'].astype(float)

In [27]:
# preview dataset
cts.head()

,TRACT,geometry
0,1011.10,"POLYGON ((-118.29793 34.26323, -118.30082 34.2..."
1,1011.22,"POLYGON ((-118.27743 34.25991, -118.27743 34.2..."
2,1012.20,"POLYGON ((-118.27818 34.25577, -118.27887 34.2..."
3,1012.21,"POLYGON ((-118.28735 34.25591, -118.28863 34.2..."
4,1012.22,"POLYGON ((-118.28594 34.2559, -118.28697 34.25..."


## Calfornia SVI Data

In [28]:
# read in CA SVI data
ca_svi = pd.read_csv('../../../data/raw/California_2020_SVI.csv')

In [29]:
# preview data
ca_svi.head()

,ST,STATE,ST_ABBR,STCNTY,COUNTY,FIPS,LOCATION,AREA_SQMI,E_TOTPOP,M_TOTPOP,...,EP_ASIAN,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE
0,6,California,CA,6001,Alameda,6001400100,"Census Tract 4001, Alameda County, California",2.681809,3035,402,...,14.0,3.0,0.0,1.3,0.0,1.3,5.6,3.9,0.6,0.9
1,6,California,CA,6001,Alameda,6001400200,"Census Tract 4002, Alameda County, California",0.226472,1983,209,...,11.0,4.7,0.3,0.4,0.0,2.0,10.6,5.0,0.4,0.6
2,6,California,CA,6001,Alameda,6001400300,"Census Tract 4003, Alameda County, California",0.428898,5058,559,...,15.3,7.6,0.1,0.3,0.7,1.1,4.1,3.7,1.1,1.4
3,6,California,CA,6001,Alameda,6001400400,"Census Tract 4004, Alameda County, California",0.276502,4179,529,...,10.0,3.3,0.6,0.8,0.0,1.0,6.7,2.7,0.1,0.2
4,6,California,CA,6001,Alameda,6001400500,"Census Tract 4005, Alameda County, California",0.228350,4021,631,...,9.6,3.9,0.0,1.0,0.0,1.0,9.4,5.2,0.0,1.0


In [30]:
# only keep observations within LA county
svi = ca_svi[ca_svi['COUNTY'] == 'Los Angeles']

In [31]:
# reset index & drop old index (keep LA County filter from above)
svi = svi.reset_index(drop=True)

In [32]:
# drop unnecessary columns
svi = svi.drop(columns = [
    'ST',
    'STATE',
    'ST_ABBR',
    'STCNTY',
    'COUNTY',
    'FIPS',
    'AREA_SQMI'
])

In [33]:
# retrieve census tract number
svi['LOCATION'] = svi['LOCATION'].astype(str)
svi['TRACT'] = svi['LOCATION'].str.extract(r'(\d+\.\d+|\d+)')
svi['TRACT'] = svi['TRACT'].astype(float)

/var/folders/cn/cp_jck311xn7c351b70mtr1h0000gn/T/ipykernel_44651/456375727.py:3: PerformanceWarning: DataFrame is highly fragmented.  This is usually the result of calling `frame.insert` many times, which has poor performance.  Consider joining all columns at once using pd.concat(axis=1) instead. To get a de-fragmented frame, use `newframe = frame.copy()`
  svi['TRACT'] = svi['LOCATION'].str.extract(r'(\d+\.\d+|\d+)')


In [34]:
# drop location (no longer needed)
svi = svi.drop(columns = ['LOCATION'])

In [35]:
# preview cleaned data
svi.head()

,E_TOTPOP,M_TOTPOP,E_HU,M_HU,E_HH,M_HH,E_POV150,M_POV150,E_UNEMP,M_UNEMP,...,MP_ASIAN,EP_AIAN,MP_AIAN,EP_NHPI,MP_NHPI,EP_TWOMORE,MP_TWOMORE,EP_OTHERRACE,MP_OTHERRACE,TRACT
0,3923,460,1629,93,1505,112,801,341,110,63,...,3.9,0.1,0.2,0.1,0.1,3.3,2.1,0.2,0.4,1011.10
1,4119,858,1406,144,1341,151,287,214,210,112,...,4.3,0.0,1.0,0.0,1.0,5.0,4.0,0.0,1.0,1011.22
2,3775,474,1462,208,1430,208,1017,310,225,112,...,6.0,0.0,1.1,0.0,1.1,2.8,2.0,0.2,0.2,1012.20
3,3787,651,1567,322,1513,325,1399,632,113,58,...,4.2,0.0,1.0,0.0,1.0,0.0,1.0,2.6,2.9,1012.21
4,2717,442,986,173,969,174,1507,486,153,161,...,5.1,0.0,1.5,0.0,1.5,0.0,1.5,0.0,1.5,1012.22


## Merging Census Tract Data & SVI Data

In [36]:
# inner join: only keep tracts present in both LA County geometry AND LA County SVI data
svi_tracts = cts.merge(svi, how='inner', on='TRACT')

print(f"Tracts with geometry + SVI data : {len(svi_tracts)}")
print(f"Null geometry rows              : {svi_tracts['geometry'].isna().sum()}")

Tracts with geometry + SVI data : 2495
Null geometry rows              : 0


In [37]:
# replace -999 values with NaN
svi_tracts = svi_tracts.replace(-999, np.nan)

In [38]:
# column names to snake case
svi_tracts = clean_columns(svi_tracts)

In [39]:
# preview cleaned data
svi_tracts.head()

,tract,geometry,e_totpop,m_totpop,e_hu,m_hu,e_hh,m_hh,e_pov_150,m_pov_150,...,ep_asian,mp_asian,ep_aian,mp_aian,ep_nhpi,mp_nhpi,ep_twomore,mp_twomore,ep_otherrace,mp_otherrace
0,1011.10,"POLYGON ((-118.29793 34.26323, -118.30082 34.2...",3923,460,1629,93,1505,112,801,341,...,10.3,3.9,0.1,0.2,0.1,0.1,3.3,2.1,0.2,0.4
1,1011.22,"POLYGON ((-118.27743 34.25991, -118.27743 34.2...",4119,858,1406,144,1341,151,287,214,...,10.3,4.3,0.0,1.0,0.0,1.0,5.0,4.0,0.0,1.0
2,1012.20,"POLYGON ((-118.27818 34.25577, -118.27887 34.2...",3775,474,1462,208,1430,208,1017,310,...,10.3,6.0,0.0,1.1,0.0,1.1,2.8,2.0,0.2,0.2
3,1012.21,"POLYGON ((-118.28735 34.25591, -118.28863 34.2...",3787,651,1567,322,1513,325,1399,632,...,7.1,4.2,0.0,1.0,0.0,1.0,0.0,1.0,2.6,2.9
4,1012.22,"POLYGON ((-118.28594 34.2559, -118.28697 34.25...",2717,442,986,173,969,174,1507,486,...,2.4,5.1,0.0,1.5,0.0,1.5,0.0,1.5,0.0,1.5


In [40]:
# drop census tracts not in continental LA county
svi_tracts_continental = svi_tracts.drop([2059, 2060])

In [41]:
# save dataset
svi_tracts_continental.to_file('../../../data/processed/svi_tracts.geojson')

## Merge SVI Tract with LA_County geometry

In [43]:
# merge svi_tract.geojson with LA County tract geometry
la_county_geojson = gpd.read_file('../../../data/raw/la_county.geojson')

la_county_geojson.head()

,name,slug,geometry
0,Acton,acton,"POLYGON ((-118.20262 34.53899, -118.18947 34.5..."
1,Adams-Normandie,adams-normandie,"POLYGON ((-118.30901 34.03741, -118.29151 34.0..."
2,Agoura Hills,agoura-hills,"POLYGON ((-118.76192 34.1682, -118.72632 34.16..."
3,Agua Dulce,agua-dulce,"POLYGON ((-118.25468 34.5583, -118.25524 34.50..."
4,Alhambra,alhambra,"POLYGON ((-118.12175 34.10504, -118.11687 34.1..."


In [44]:
# align CRS before spatial merge
if la_county_geojson.crs != svi_tracts_continental.crs:
    la_county_geojson = la_county_geojson.to_crs(svi_tracts_continental.crs)

# reproject to projected CRS for accurate centroids, then assign each tract to a neighborhood
svi_centroids = svi_tracts_continental.copy()
svi_centroids['geometry'] = svi_tracts_continental.to_crs('EPSG:3310').centroid.to_crs(svi_tracts_continental.crs)

tract_to_nbhd = gpd.sjoin(
    svi_centroids,
    la_county_geojson[['name', 'slug', 'geometry']],
    how='left',
    predicate='within'
).drop(columns='index_right')

# deduplicate in case any centroid lands on a shared boundary
tract_to_nbhd = tract_to_nbhd[~tract_to_nbhd.index.duplicated(keep='first')]

# drop tracts not within any neighborhood
tract_to_nbhd = tract_to_nbhd.dropna(subset=['name'])

print(f"Tracts assigned to a neighborhood : {len(tract_to_nbhd)}")
print(f"Tracts dropped (no neighborhood)  : {len(svi_tracts_continental) - len(tract_to_nbhd)}")

# --- aggregate to neighborhood level ---
# raw count columns (e_ but not ep_): sum
e_cols  = [c for c in tract_to_nbhd.columns if c.startswith('e_') and not c.startswith('ep_')]
# percentage columns (ep_, rpl_, spl_, f_): population-weighted mean
pct_cols = [c for c in tract_to_nbhd.columns if c.startswith('ep_') or c.startswith('rpl_') or c.startswith('spl_') or c.startswith('f_')]
# count margins of error (m_ but not mp_): root-sum-of-squares
m_cols  = [c for c in tract_to_nbhd.columns if c.startswith('m_') and not c.startswith('mp_')]
# percentage margins of error (mp_): mean
mp_cols = [c for c in tract_to_nbhd.columns if c.startswith('mp_')]

agg_e  = tract_to_nbhd.groupby('name')[e_cols].sum()
agg_m  = tract_to_nbhd.groupby('name')[m_cols].agg(lambda x: np.sqrt((x ** 2).sum()))
agg_mp = tract_to_nbhd.groupby('name')[mp_cols].mean()

pop      = tract_to_nbhd['e_totpop'].replace(0, np.nan)
weighted = tract_to_nbhd[pct_cols].multiply(pop, axis=0)
agg_pct  = (
    weighted.groupby(tract_to_nbhd['name']).sum()
    .div(tract_to_nbhd.groupby('name')['e_totpop'].sum(), axis=0)
)

# attach neighborhood geometry from la_county_geojson
neighborhood_geom = la_county_geojson.set_index('name')[['slug', 'geometry']]
la_county_svi = (
    neighborhood_geom
    .join(agg_e)
    .join(agg_pct)
    .join(agg_m)
    .join(agg_mp)
    .reset_index()
)
la_county_svi = gpd.GeoDataFrame(la_county_svi, geometry='geometry', crs=svi_tracts_continental.crs)

print(f"Neighborhoods in output : {len(la_county_svi)}")
la_county_svi.to_file('../../../data/processed/la_county_svi.geojson', driver='GeoJSON')

Tracts assigned to a neighborhood : 2485
Tracts dropped (no neighborhood)  : 8
Neighborhoods in output : 272
